# 第十五章 构建赛博小镇

这一章，我们将探索一个全新的方向：**将智能体技术与游戏引擎结合，构建一个充满生命力的 AI 小镇**。

还记得《模拟人生》或《动物森友会》中那些栩栩如生的 NPC 吗？他们有自己的性格、记忆和社交关系。本章的赛博小镇将是一个类似的项目，但与传统游戏不同的是，我们的 NPC 拥有真正的
——他们能够理解玩家的对话，记住过去的互动，并根据好感度做出不同的反应。本章的赛博小镇包含以下核心功能：

**（1）智能 NPC 对话系统**：玩家可以与 NPC 进行自然语言对话，NPC 会根据自己的角色设定和记忆做出回应。

**（2）记忆系统**：NPC 拥有短期记忆和长期记忆，能够记住与玩家的互动历史。

**（3）好感度系统**：NPC 对玩家的态度会随着互动而变化，从陌生到熟悉，从友好到亲密。

**（4）游戏化交互**：玩家可以在 2D 像素风格的办公室场景中自由移动，与不同的 NPC 互动。

**（5）实时日志系统**：所有对话和互动都会被记录，方便调试和分析。

## 15.1 项目概述与架构设计

### 15.1.1 为什么要构建 AI 小镇

传统游戏中的 NPC 通常只能说固定的台词，或者通过预设的对话树进行有限的互动。即使是最复杂的 RPG 游戏，NPC 的对话也是由编剧事先写好的。这种方式虽然可控，但缺乏真正的
和
。

想象一下，如果游戏中的 NPC 能够理解你说的任何话，不再局限于预设的选项，你可以用自然语言与 NPC 交流。NPC 会记得你上次说了什么，你们的关系如何，甚至你的喜好。每个 NPC 都有自己的职业、性格和说话风格。NPC 对你的态度会随着互动而变化，从陌生人到朋友，甚至挚友。

这就是 AI 技术为游戏带来的新可能：在教育游戏中，NPC 可以扮演历史人物与学生进行互动式教学；在虚拟办公室中，NPC 可以扮演同事、导师，提供帮助和建议；还可以应用于心理健康、传统游戏 AI NPC 等场景。

### 15.1.2 技术架构概览

赛博小镇采用**游戏引擎+后端服务**的分离架构，分为四个层次（图 15.1）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-1.png" alt="" width="85%"/>
  <p>图 15.1 赛博小镇技术架构</p>
</div>

- **前端层（Godot 4.5）**：游戏渲染、玩家控制、NPC 显示、对话 UI
- **后端层（FastAPI）**：API 路由、NPC 状态管理、对话处理、日志记录
- **智能体层（HelloAgents）**：NPC 智能、记忆管理、好感度计算；每个 NPC 是一个 SimpleAgent 实例
- **外部服务层**：LLM API、Qdrant 向量数据库、SQLite 关系数据库

数据流转过程（图 15.2）：玩家按 E 键 → Godot 发 HTTP 请求 → FastAPI 调用 Agent → 检索记忆 → 调用 LLM 生成回复 → 更新好感度 → 返回 Godot → 显示 NPC 回复。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-2.png" alt="" width="85%"/>
  <p>图 15.2 数据流转过程</p>
</div>

In [ ]:
# 项目目录结构
project_structure = """
Helloagents-AI-Town/
├── helloagents-ai-town/           # Godot游戏项目
│   ├── project.godot              # Godot项目配置
│   ├── scenes/                    # 游戏场景
│   │   ├── main.tscn              # 主场景(办公室)
│   │   ├── player.tscn            # 玩家角色
│   │   ├── npc.tscn               # NPC角色
│   │   └── dialogue_ui.tscn       # 对话UI
│   ├── scripts/                   # GDScript脚本
│   │   ├── main.gd                # 主场景逻辑
│   │   ├── player.gd              # 玩家控制
│   │   ├── npc.gd                 # NPC行为
│   │   ├── dialogue_ui.gd         # 对话UI逻辑
│   │   ├── api_client.gd          # API客户端
│   │   └── config.gd              # 配置管理
│   └── assets/                    # 游戏资源
│       ├── characters/            # 角色精灵图
│       ├── interiors/             # 室内场景
│       ├── ui/                    # UI素材
│       └── audio/                 # 音效音乐
│
└── backend/                       # Python后端
    ├── main.py                    # FastAPI主程序
    ├── agents.py                  # NPC Agent系统
    ├── relationship_manager.py    # 好感度管理
    ├── state_manager.py           # 状态管理
    ├── logger.py                  # 日志系统
    ├── config.py                  # 配置管理
    ├── models.py                  # 数据模型
    ├── requirements.txt           # Python依赖
    └── .env.example               # 环境变量示例
"""
print(project_structure)

### 15.1.3 快速体验：5 分钟运行项目

**环境要求**：Godot 4.2+、Python 3.10+、LLM API 密钥（OpenAI/DeepSeek/智谱等）

项目位于 `code/chapter15/Helloagents-AI-Town`。

In [ ]:
# 启动说明（在终端中执行，不可直接在此运行）
startup_backend = """
# 1. 进入backend目录
cd Helloagents-AI-Town/backend

# 2. 安装依赖
pip install -r requirements.txt

# 3. 配置环境变量
cp .env.example .env
# 编辑.env文件，填写你的API密钥

# 4. 启动后端服务
python main.py
# 成功后输出：
# ============================================================
# 🎮 赛博小镇后端服务启动中...
# ============================================================
# ✅ 所有服务已启动!
# 📡 API地址: http://0.0.0.0:8000
# 📚 API文档: http://0.0.0.0:8000/docs
# ============================================================
"""

startup_godot = """
# Godot 直接在官网下载即可（Windows .exe / Mac .dmg）
# Windows: https://godotengine.org/download/windows/
# Mac:     https://godotengine.org/download/macos/

# 1. 打开 Godot 引擎
# 2. 点击
按钮
# 3. 浏览到 Helloagents-AI-Town/helloagents-ai-town/scenes/main.tscn
# 4. 点击

# 5. 按 F5 或点击
启动游戏
"""

print("后端启动命令:", startup_backend)
print("Godot启动说明:", startup_godot)

游戏启动后，你会看到一个像素风格的 Datawhale 办公室场景（图 15.3）。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-3.png" alt="" width="85%"/>
  <p>图 15.3 赛博小镇游戏场景</p>
</div>

使用 WASD 键移动，走到 NPC 附近按 E 键即可对话（图 15.4）。NPC 会根据角色设定和互动历史做出回应，好感度从
逐步提升到
。好感度变化详情记录在 `backend/logs/dialogue_YYYY-MM-DD.log` 中。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-4.png" alt="" width="85%"/>
  <p>图 15.4 与 NPC 对话界面</p>
</div>

## 15.2 NPC 智能体系统

### 15.2.1 基于 HelloAgents 的 SimpleAgent

在赛博小镇中，每个 NPC 都是一个独立的智能体。我们使用 HelloAgents 框架中的 `SimpleAgent` 来实现 NPC 的智能——它封装了 LLM 调用、消息管理等核心功能。

NPC Agent 的工作流程（图 15.5）：接收玩家消息 → 检索短期+长期记忆 → 结合角色 Prompt → 调用 LLM → 返回回复 → 更新记忆。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-5.png" alt="" width="85%"/>
  <p>图 15.5 NPC Agent 工作流程</p>
</div>

In [ ]:
# 创建 NPC Agent（后端项目中运行）
create_npc_code = '''
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.memory import MemoryManager, WorkingMemory, EpisodicMemory

def create_npc_agent(npc_id: str, name: str, role: str, personality: str):
    """创建NPC Agent"""
    system_prompt = f"""你是{name},一位{role}。
你的性格特点:{personality}

你在Datawhale办公室工作,与同事们一起推动开源社区的发展。
请根据你的角色和性格,自然地与玩家对话。
记住你们之前的对话内容,保持对话的连贯性。
"""

    llm = HelloAgentsLLM()

    # WorkingMemory: 短期记忆（容量10条，保留120分钟）
    # EpisodicMemory: 长期记忆（SQLite + Qdrant向量数据库）
    memory_manager = MemoryManager(
        working_memory=WorkingMemory(capacity=10, ttl_minutes=120),
        episodic_memory=EpisodicMemory(
            db_path=f"memory_data/{npc_id}_episodic.db",
            collection_name=f"{npc_id}_memories"
        )
    )

    agent = SimpleAgent(
        name=name,
        llm=llm,
        system_prompt=system_prompt,
        memory_manager=memory_manager
    )
    return agent
'''
print("NPC Agent 创建代码:")
print(create_npc_code)

### 15.2.2 NPC 角色设定与 Prompt 设计

三个 NPC 各有鲜明的性格：

In [ ]:
# 三个 NPC 角色配置
npc_zhang = {
    "npc_id": "zhang_san",
    "name": "张三",
    "role": "Python工程师",
    "personality": "严谨、专业、喜欢分享技术知识。说话直接,注重代码质量。"
}

npc_li = {
    "npc_id": "li_si",
    "name": "李四",
    "role": "产品经理",
    "personality": "外向、善于沟通、注重用户体验。喜欢从用户角度思考问题。"
}

npc_wang = {
    "npc_id": "wang_wu",
    "name": "王五",
    "role": "UI设计师",
    "personality": "温和、富有创意、审美独特。注重视觉呈现和用户体验。"
}

npcs = [npc_zhang, npc_li, npc_wang]
print("NPC 角色配置:")
for npc in npcs:
    print(f"  [{npc['name']}] {npc['role']}：{npc['personality'][:20]}...")

### 15.2.3 记忆系统集成

记忆系统是 NPC 智能的关键（图 15.6）：
- **短期记忆（WorkingMemory）**：存储最近对话，保持上下文连贯性
- **长期记忆（EpisodicMemory）**：存储所有历史，支持语义检索（
）

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-6.png" alt="" width="85%"/>
  <p>图 15.6 记忆系统架构</p>
</div>

In [ ]:
# Agent 处理对话的流程
def process_dialogue(agent, player_message: str):
    """（伪代码）展示记忆系统集成流程"""

    # 1. 从短期记忆获取最近对话
    # recent_messages = agent.memory_manager.working_memory.get_recent_messages(5)
    recent_messages = ["上次我们讨论了Python性能优化"]  # 模拟

    # 2. 从长期记忆检索相关历史
    # relevant_memories = agent.memory_manager.episodic_memory.search(
    #     query=player_message, top_k=3
    # )
    relevant_memories = ["三周前：讨论过装饰器用法"]  # 模拟

    context = {"recent": recent_messages, "relevant": relevant_memories}
    print(f"玩家消息: {player_message}")
    print(f"短期记忆: {recent_messages}")
    print(f"长期记忆相关内容: {relevant_memories}")

    # 3. 调用Agent生成回复（传入上下文）
    # reply = agent.run(player_message, context=context)
    reply = "[模拟] 是的，性能优化很重要！上次我们聊到了装饰器……"

    # 4. 保存到记忆系统
    # agent.memory_manager.add_interaction(player_message, reply)

    return reply


# 演示
reply = process_dialogue(None, "能帮我看看这段Python代码吗？")
print(f"\nNPC 回复: {reply}")

### 15.2.4 批量对话生成：轻负载模式

**问题**：多个 NPC 并发 LLM 调用成本高、延迟大。

**解决方案**：批量生成系统——一次 LLM 调用生成所有 NPC 的背景对话，成本降至原来的 1/3。

批量生成 vs 传统模式（图 15.7）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-7.png" alt="" width="85%"/>
  <p>图 15.7 批量生成 vs 传统模式</p>
</div>

In [ ]:
import json
from typing import Dict, Optional
from datetime import datetime


NPC_ROLES = {
    "张三": {"title": "Python工程师", "location": "工位上", "activity": "写代码", "personality": "严谨专业"},
    "李四": {"title": "产品经理",    "location": "会议室", "activity": "看文档", "personality": "外向善谈"},
    "王五": {"title": "UI设计师",    "location": "设计区", "activity": "画原型", "personality": "温和创意"},
}


class NPCBatchGenerator:
    """批量生成NPC对话的生成器"""

    def __init__(self):
        # self.llm = HelloAgentsLLM()
        self.npc_configs = NPC_ROLES

    def _get_current_context(self) -> str:
        hour = datetime.now().hour
        if 9 <= hour < 12:   return "上午工作时间"
        elif 12 <= hour < 14: return "午餐时间"
        elif 14 <= hour < 18: return "下午工作时间"
        else:                 return "下班时间"

    def _build_batch_prompt(self, context: Optional[str] = None) -> str:
        if context is None:
            context = self._get_current_context()

        npc_descriptions = [
            f"- {name}({cfg['title']}): 在{cfg['location']}{cfg['activity']},性格{cfg['personality']}"
            for name, cfg in self.npc_configs.items()
        ]

        return f"""请为Datawhale办公室的3个NPC生成当前的对话或行为描述。

【场景】{context}

【NPC信息】
{chr(10).join(npc_descriptions)}

【生成要求】只返回JSON，格式如下示例:
{{"张三": "这个bug真是见鬼了...", "李四": "功能优先级需要重新评估。", "王五": "这杯咖啡真不错，灵感来了!"}}
"""

    def generate_batch_dialogues(self, context: Optional[str] = None) -> Dict[str, str]:
        """批量生成所有NPC的对话（模拟）"""
        prompt = self._build_batch_prompt(context)
        # 实际项目中：response = self.llm.invoke([...])
        # 模拟 LLM 返回
        mock_response = '{"张三": "这个类型提示写得不对，要改一下。", "李四": "用户反馈里有个好需求，值得做。", "王五": "配色方案定了，清新又专业！"}'
        return json.loads(mock_response)


# 演示批量生成
generator = NPCBatchGenerator()
dialogues = generator.generate_batch_dialogues(context="下午工作时间")
print("批量生成结果（一次LLM调用，3个NPC）:")
for name, dialogue in dialogues.items():
    print(f"  {name}: {dialogue}")
print(f"\n构建的Prompt:\n{generator._build_batch_prompt('下午工作时间')}")

**混合模式：批量生成 + 即时响应**

| 场景 | 模式 | 说明 |
|------|------|------|
| 玩家未交互时 | 批量生成（后台定时） | NPC 头顶显示背景对话气泡，显得
 |
| 玩家按 E 键交互 | 即时响应（专属 Agent） | 根据玩家消息和历史记忆生成个性化回复 |

In [ ]:
# 混合模式实现（后端 main.py 中的关键代码）
hybrid_mode_code = '''
import asyncio

# --- 即时响应：玩家交互 ---
@app.post("/dialogue")
async def dialogue(request: DialogueRequest):
    """处理玩家与NPC的对话(即时响应)"""
    agent = npc_agents.get(request.npc_id)
    if not agent:
        raise HTTPException(status_code=404, detail="NPC not found")

    # 调用专属Agent生成个性化回复（不使用批量生成）
    reply = agent.run(request.player_message)

    affinity_change = relationship_manager.update_affinity(
        request.npc_id, request.player_name, request.player_message, reply
    )
    return {"npc_reply": reply, "affinity_score": affinity_change["score"],
            "affinity_level": affinity_change["level"]}

# --- 批量生成：后台定时任务 ---
async def background_dialogue_update():
    """每5分钟批量更新NPC背景对话"""
    while True:
        try:
            batch_generator = get_batch_generator()
            dialogues = batch_generator.generate_batch_dialogues()
            for npc_name, dialogue in dialogues.items():
                state_manager.update_npc_background_dialogue(npc_name, dialogue)
            print(f"✅ 背景对话更新完成: {len(dialogues)}个NPC")
        except Exception as e:
            print(f"❌ 背景对话更新失败: {e}")
        await asyncio.sleep(300)  # 等待5分钟
'''
print("混合模式代码:")
print(hybrid_mode_code)

## 15.3 好感度系统设计

### 15.3.1 好感度等级划分

五级好感度系统（图 15.8）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-8.png" alt="" width="85%"/>
  <p>图 15.8 好感度等级划分</p>
</div>

| 等级 | 分数范围 | NPC 行为 |
|------|----------|----------|
| 陌生 | 0-20 | 礼貌但疏远，回复简短 |
| 熟悉 | 21-40 | 开始记住玩家，交流更自然 |
| 友好 | 41-60 | 视作朋友，主动分享信息 |
| 亲密 | 61-80 | 非常信任，分享私人话题 |
| 挚友 | 81-100 | 无话不谈，回复充满热情 |

### 15.3.2 好感度计算逻辑

使用 LLM 分析对话情感，自动判断玩家态度（友好/中立/不友好），动态调整分数（图 15.9）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-9.png" alt="" width="85%"/>
  <p>图 15.9 好感度计算流程</p>
</div>

In [ ]:
from typing import Dict


class RelationshipManager:
    """好感度管理器"""

    def __init__(self):
        self.affinity_data: Dict[str, dict] = {}
        # self.llm = HelloAgentsLLM()

    def analyze_sentiment(self, player_message: str, npc_reply: str) -> int:
        """（模拟）用 LLM 分析对话情感，返回好感度变化值"""
        # 实际项目中调用 LLM 判断：友好(+5) / 中立(+2) / 不友好(-3)
        # prompt = f"分析玩家态度...\n玩家: {player_message}\nNPC: {npc_reply}\n只返回数字"
        # response = self.llm.think([{"role": "user", "content": prompt}])
        # return max(-3, min(5, int(response.strip())))

        # 简单模拟：含
/
则友好，含
则不友好
        msg = player_message.lower()
        if any(w in msg for w in ["谢谢", "你好", "棒", "太好了"]):
            return 5  # 友好
        elif any(w in msg for w in ["烦", "讨厌", "不想"]):
            return -3  # 不友好
        return 2  # 中立

    def get_affinity_level(self, score: int) -> str:
        if score <= 20:  return "陌生"
        elif score <= 40: return "熟悉"
        elif score <= 60: return "友好"
        elif score <= 80: return "亲密"
        else:             return "挚友"

    def get_affinity(self, npc_id: str, player_name: str) -> dict:
        key = f"{npc_id}_{player_name}"
        return self.affinity_data.get(key, {"score": 0, "level": "陌生", "interaction_count": 0})

    def update_affinity(self, npc_id: str, player_name: str,
                        player_message: str, npc_reply: str) -> dict:
        """更新好感度"""
        key = f"{npc_id}_{player_name}"
        if key not in self.affinity_data:
            self.affinity_data[key] = {"score": 0, "level": "陌生", "interaction_count": 0}

        score_change = self.analyze_sentiment(player_message, npc_reply)
        current = self.affinity_data[key]
        new_score = max(0, min(100, current["score"] + score_change))

        self.affinity_data[key] = {
            "score": new_score,
            "level": self.get_affinity_level(new_score),
            "interaction_count": current["interaction_count"] + 1
        }
        return self.affinity_data[key]


# 演示好感度系统
rm = RelationshipManager()

interactions = [
    ("你好，张三！能介绍一下你自己吗？",       "你好！我是张三，负责框架核心开发。"),
    ("谢谢你的介绍！你对Python很熟悉对吗？", "是的，我用Python已经7年了。"),
    ("太棒了！能教我一些Python技巧吗？",       "当然可以，你想了解哪方面？"),
    ("普通的问题，今天天气怎么样？",           "天气还不错。"),
]

print("好感度变化演示:")
for player_msg, npc_reply in interactions:
    result = rm.update_affinity("zhang_san", "player1", player_msg, npc_reply)
    sentiment = rm.analyze_sentiment(player_msg, npc_reply)
    print(f"  [{result['level']:3s}] 分数={result['score']:3d} 变化={'+' if sentiment>0 else ''}{sentiment} | 玩家: {player_msg[:15]}...")

### 15.3.3 好感度影响对话

好感度通过动态调整 NPC 的系统提示词影响回复风格：

In [ ]:
def create_npc_agent_with_affinity(
    name: str, role: str, personality: str, affinity_level: str
):
    """根据好感度动态生成 NPC 系统 Prompt"""
    affinity_prompts = {
        "陌生": "你刚认识这位玩家,保持礼貌但不要过于热情。回复简短专业。",
        "熟悉": "你已经认识这位玩家,可以进行正常的交流。回复自然友好。",
        "友好": "你把这位玩家当作朋友,愿意分享更多信息。回复详细热情。",
        "亲密": "你非常信任这位玩家,可以分享私人话题。回复充满关心。",
        "挚友": "你把这位玩家当作最好的朋友,无话不谈。回复亲切真诚。"
    }

    system_prompt = f"""你是{name},一位{role}。
你的性格特点:{personality}

当前与玩家的关系:{affinity_level}
{affinity_prompts.get(affinity_level, affinity_prompts['陌生'])}

你在Datawhale办公室工作，请根据你的角色、性格和与玩家的关系自然地回复。
"""
    return system_prompt  # 实际项目中返回 SimpleAgent 实例


# 对比不同好感度等级的 Prompt
for level in ["陌生", "友好", "挚友"]:
    prompt = create_npc_agent_with_affinity("张三", "Python工程师", "严谨专业", level)
    lines = prompt.strip().split("\n")
    print(f"\n[{level}] Prompt 第5行（行为指令）:")
    print(f"  → {lines[4]}")

## 15.4 后端服务实现

### 15.4.1 FastAPI 应用结构

后端采用模块化设计（图 15.10）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-10.png" alt="" width="85%"/>
  <p>图 15.10 后端应用结构</p>
</div>

In [ ]:
# FastAPI 主程序（main.py）
main_py_code = '''
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

from agents import NPCAgentManager
from relationship_manager import RelationshipManager
from state_manager import StateManager
from logger import DialogueLogger
from config import settings

app = FastAPI(
    title="赛博小镇后端服务",
    description="基于HelloAgents的AI NPC对话系统",
    version="1.0.0"
)

# 配置CORS（允许Godot前端跨域访问）
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 初始化各个管理器
agent_manager = NPCAgentManager()
relationship_manager = RelationshipManager()
state_manager = StateManager()
dialogue_logger = DialogueLogger()

@app.on_event("startup")
async def startup_event():
    print("=" * 60)
    print("🎮 赛博小镇后端服务启动中...")
    print("=" * 60)
    agent_manager.initialize_npcs()
    state_manager.initialize_npcs()
    print("✅ 所有服务已启动!")
    print(f"📡 API地址: http://0.0.0.0:{settings.PORT}")
    print("=" * 60)

@app.get("/")
async def root():
    return {"status": "running", "message": "赛博小镇后端服务正在运行",
            "version": "1.0.0", "npcs": state_manager.get_npc_count()}

if __name__ == "__main__":
    uvicorn.run(app, host=settings.HOST, port=settings.PORT, log_level="info")
'''
print("FastAPI main.py:")
print(main_py_code)

### 15.4.2 API 路由设计

核心 API 端点（图 15.11）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-11.png" alt="" width="85%"/>
  <p>图 15.11 API 调用流程</p>
</div>

In [ ]:
# API 路由（添加到 main.py）
api_routes_code = '''
# --- 获取NPC状态 ---
@app.get("/npcs/status")
async def get_npc_status():
    """获取所有NPC的状态"""
    return {"npcs": state_manager.get_all_npc_states()}

# --- 核心对话接口 ---
@app.post("/dialogue")
async def dialogue(request: DialogueRequest):
    """处理玩家与NPC的对话（含好感度更新和日志）"""
    if not agent_manager.has_npc(request.npc_id):
        raise HTTPException(status_code=404, detail=f"NPC {request.npc_id} 不存在")
    if state_manager.is_npc_busy(request.npc_id):
        raise HTTPException(status_code=409, detail="NPC 正在与其他玩家对话")

    state_manager.set_npc_busy(request.npc_id, True)
    try:
        affinity_info = relationship_manager.get_affinity(request.npc_id, request.player_name)
        agent = agent_manager.get_agent(request.npc_id, affinity_info["level"])
        reply = agent.run(request.player_message)
        new_affinity = relationship_manager.update_affinity(
            request.npc_id, request.player_name, request.player_message, reply
        )
        dialogue_logger.log_dialogue(
            npc_id=request.npc_id, player_name=request.player_name,
            player_message=request.player_message, npc_reply=reply,
            affinity_info=new_affinity
        )
        return DialogueResponse(
            npc_reply=reply,
            affinity_level=new_affinity["level"],
            affinity_score=new_affinity["score"]
        )
    finally:
        state_manager.set_npc_busy(request.npc_id, False)

# --- 好感度查询 ---
@app.get("/affinity/{npc_id}/{player_name}")
async def get_affinity(npc_id: str, player_name: str):
    return relationship_manager.get_affinity(npc_id, player_name)
'''
print("API 路由代码:")
print(api_routes_code)

### 15.4.3 状态管理与日志系统

In [ ]:
from typing import List, Optional
from datetime import datetime


class StateManager:
    """NPC状态管理器（防止并发冲突）"""

    def __init__(self):
        self.npc_states: dict = {}

    def initialize_npcs(self):
        npcs = [
            {"npc_id": "zhang_san", "name": "张三", "role": "Python工程师", "position": {"x": 300, "y": 200}},
            {"npc_id": "li_si",    "name": "李四", "role": "产品经理",    "position": {"x": 500, "y": 200}},
            {"npc_id": "wang_wu",  "name": "王五", "role": "UI设计师",   "position": {"x": 700, "y": 200}},
        ]
        for npc in npcs:
            self.npc_states[npc["npc_id"]] = {
                **npc, "is_busy": False, "current_action": "idle", "last_interaction": None
            }

    def is_npc_busy(self, npc_id: str) -> bool:
        npc = self.npc_states.get(npc_id)
        return npc["is_busy"] if npc else False

    def set_npc_busy(self, npc_id: str, busy: bool):
        if npc_id in self.npc_states:
            self.npc_states[npc_id]["is_busy"] = busy
            if busy:
                self.npc_states[npc_id]["last_interaction"] = datetime.now().isoformat()

    def get_npc_count(self) -> int:
        return len(self.npc_states)


# 测试状态管理
sm = StateManager()
sm.initialize_npcs()
print(f"NPC数量: {sm.get_npc_count()}")
print(f"张三是否忙碌: {sm.is_npc_busy('zhang_san')}")
sm.set_npc_busy("zhang_san", True)
print(f"设为忙碌后: {sm.is_npc_busy('zhang_san')}")
sm.set_npc_busy("zhang_san", False)
print(f"释放后: {sm.is_npc_busy('zhang_san')}")

In [ ]:
import logging
from pathlib import Path


class DialogueLogger:
    """对话日志记录器（双输出：控制台+文件）"""

    def __init__(self, log_dir: str = "logs"):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)

        today = datetime.now().strftime("%Y-%m-%d")
        log_file = self.log_dir / f"dialogue_{today}.log"

        self.logger = logging.getLogger("DialogueLogger")
        if not self.logger.handlers:  # 防止重复添加
            self.logger.setLevel(logging.INFO)
            # 控制台处理器
            ch = logging.StreamHandler()
            ch.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', '%H:%M:%S'))
            self.logger.addHandler(ch)
            # 文件处理器
            fh = logging.FileHandler(log_file, encoding='utf-8')
            fh.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
            self.logger.addHandler(fh)

    def log_dialogue(self, npc_id: str, player_name: str,
                     player_message: str, npc_reply: str, affinity_info: dict):
        msg = (f"NPC:{npc_id} 玩家:{player_name} | "
               f"好感度:{affinity_info['level']}({affinity_info['score']}/100) | "
               f"玩家:{player_message[:20]}... | NPC:{npc_reply[:20]}...")
        self.logger.info(msg)

    def log_error(self, error_message: str):
        self.logger.error(error_message)


# 测试日志
logger = DialogueLogger()
logger.log_dialogue(
    npc_id="zhang_san",
    player_name="player1",
    player_message="你好！能教我Python装饰器吗？",
    npc_reply="当然可以！装饰器是Python最优雅的特性之一……",
    affinity_info={"level": "友好", "score": 55, "interaction_count": 8}
)

### 15.4.4 理解 Godot 的场景系统

**节点（Node）**：Godot 最基本的构建块，类似乐高积木，每种节点专注一件事：
- `Sprite2D` → 显示图片；`CharacterBody2D` → 角色物理移动；`Area2D` → 碰撞检测区域

**场景（Scene）**：一组节点的集合（`.tscn` 文件），可复用、可嵌套。赛博小镇中三个 NPC 都是同一个 `NPC.tscn` 的实例，修改模板自动影响所有 NPC。

玩家场景节点树示例：
```
Player (CharacterBody2D)   ← 根节点，负责物理移动
├─ AnimatedSprite2D        ← 角色动画
├─ CollisionShape2D        ← 碰撞形状
└─ Camera2D                ← 摄像机跟随
```

## 15.5 Godot 游戏场景构建

**为什么选择 Godot？**
1. **2D 天然优势**：`TileMap`、`AnimatedSprite2D`、`CharacterBody2D` 等节点专为 2D 设计
2. **完全开源免费**：MIT 许可证，无版权费用，商业化无障碍
3. **学习成本极低**：GDScript 类似 Python，已有 Python 基础几乎零门槛
4. **与 Python 后端集成简单**：内置 `HTTPRequest` 节点直连 FastAPI

### 15.5.1 场景设计与资源组织

四个核心场景（图 15.12）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-12.png" alt="" width="85%"/>
  <p>图 15.12 赛博小镇的四个核心场景</p>
</div>

- **Main**：主场景，实例化玩家、3 个 NPC、对话 UI、背景音乐
- **Player**：玩家角色（CharacterBody2D + AnimatedSprite2D + Camera2D）
- **NPC**：通用 NPC 模板（张三/李四/王五均为此场景实例）
- **DialogueUI**：CanvasLayer + Panel + 输入框/按钮

### 15.5.2 玩家控制实现

玩家控制脚本 `player.gd`（GDScript，类 Python 语法）：

In [ ]:
# player.gd — GDScript 代码（与 Python 高度相似）
player_gd_code = '''
extends CharacterBody2D

@export var speed: float = 200.0  # 移动速度

var nearby_npc: Node = null        # 当前可交互的NPC
var is_interacting: bool = false   # 交互时禁用移动

@onready var animated_sprite: AnimatedSprite2D = $AnimatedSprite2D
@onready var camera: Camera2D = $Camera2D

func _ready():
    add_to_group("player")  # 重要！NPC通过此group识别玩家
    camera.enabled = true
    animated_sprite.play("idle")

func _physics_process(_delta: float):
    if is_interacting:             # 交互时停止移动
        velocity = Vector2.ZERO
        move_and_slide()
        return

    var input_direction = Input.get_vector("ui_left", "ui_right", "ui_up", "ui_down")
    velocity = input_direction * speed
    move_and_slide()
    update_animation(input_direction)

func update_animation(direction: Vector2):
    """根据移动方向播放4方向动画"""
    if direction.length() > 0:
        if abs(direction.x) > abs(direction.y):
            animated_sprite.play("walk_right" if direction.x > 0 else "walk_left")
        else:
            animated_sprite.play("walk_down" if direction.y > 0 else "walk_up")
    else:
        animated_sprite.play("idle")

func _input(event: InputEvent):
    # 按E键/Enter键与附近NPC交互
    if event is InputEventKey and event.pressed and not event.echo:
        if event.keycode == KEY_E or event.keycode == KEY_ENTER:
            if nearby_npc != null:
                get_tree().call_group("dialogue_system", "start_dialogue", nearby_npc.npc_name)

func set_nearby_npc(npc: Node):
    nearby_npc = npc

func set_interacting(interacting: bool):
    is_interacting = interacting
'''
print("player.gd (GDScript):\n" + player_gd_code)

### 15.5.3 NPC 行为与交互

NPC 实现三个核心功能：随机巡逻游走、检测玩家进入交互范围、显示对话气泡。

In [ ]:
# npc.gd — NPC 脚本（GDScript）
npc_gd_code = '''
extends CharacterBody2D

@export var npc_name: String = "张三"
@export var npc_title: String = "Python工程师"
@export var move_speed: float = 50.0
@export var wander_range: float = 200.0      # 巡逻范围
@export var wander_interval_min: float = 3.0  # 最小巡逻间隔（秒）
@export var wander_interval_max: float = 8.0  # 最大巡逻间隔（秒）

var wander_target: Vector2 = Vector2.ZERO
var wander_timer: float = 0.0
var is_wandering: bool = false
var is_interacting: bool = false
var spawn_position: Vector2 = Vector2.ZERO

@onready var interaction_area: Area2D = $InteractionArea
@onready var name_label: Label = $NameLabel
@onready var dialogue_label: Label = $DialogueLabel

func _ready():
    add_to_group("npcs")
    name_label.text = npc_name
    dialogue_label.visible = false
    spawn_position = global_position
    # 连接玩家进入/离开交互范围的信号
    interaction_area.body_entered.connect(_on_body_entered)
    interaction_area.body_exited.connect(_on_body_exited)
    choose_new_wander_target()

func _on_body_entered(body: Node2D):
    """玩家进入交互范围"""
    if body.is_in_group("player"):
        body.set_nearby_npc(self)  # 通知玩家当前可交互NPC是自己

func _on_body_exited(body: Node2D):
    """玩家离开交互范围"""
    if body.is_in_group("player"):
        body.set_nearby_npc(null)

func _physics_process(delta: float):
    if is_interacting:
        velocity = Vector2.ZERO
        move_and_slide()
        return

    wander_timer -= delta
    if wander_timer <= 0:
        choose_new_wander_target()
        wander_timer = randf_range(wander_interval_min, wander_interval_max)

    if is_wandering:
        if global_position.distance_to(wander_target) < 10:
            is_wandering = false
            velocity = Vector2.ZERO
        else:
            var direction = (wander_target - global_position).normalized()
            velocity = direction * move_speed
        move_and_slide()

func choose_new_wander_target():
    """在出生位置附近随机选择巡逻目标"""
    var offset = Vector2(
        randf_range(-wander_range, wander_range),
        randf_range(-wander_range, wander_range)
    )
    wander_target = spawn_position + offset
    is_wandering = true

func update_dialogue(dialogue: String):
    """更新头顶对话气泡（10秒后消失）"""
    dialogue_label.text = dialogue
    dialogue_label.visible = true
    await get_tree().create_timer(10.0).timeout
    dialogue_label.visible = false

func set_interacting(interacting: bool):
    is_interacting = interacting
'''
print("npc.gd (GDScript):\n" + npc_gd_code)

## 15.6 前后端通信实现

### 15.6.1 API 客户端封装

Godot 使用 `HTTPRequest` 节点发送异步 HTTP 请求，通过**信号机制**通知响应结果（保证游戏不卡顿）。

In [ ]:
# api_client.gd — API 客户端（设为 AutoLoad 单例）
api_client_gd_code = '''
extends Node

# 信号定义（异步响应通知）
signal chat_response_received(npc_name: String, message: String)
signal chat_error(error_message: String)
signal npc_status_received(dialogues: Dictionary)

var http_chat: HTTPRequest
var http_status: HTTPRequest

func _ready():
    http_chat = HTTPRequest.new()
    http_status = HTTPRequest.new()
    add_child(http_chat)
    add_child(http_status)
    http_chat.request_completed.connect(_on_chat_request_completed)
    http_status.request_completed.connect(_on_status_request_completed)

func send_chat(npc_name: String, message: String) -> void:
    """发送对话请求（异步）"""
    var data = {"npc_name": npc_name, "message": message}
    var headers = ["Content-Type: application/json"]
    http_chat.request(
        Config.API_CHAT,
        headers,
        HTTPClient.METHOD_POST,
        JSON.stringify(data)
    )

func _on_chat_request_completed(_result, response_code, _headers, body):
    if response_code != 200:
        chat_error.emit("服务器错误: " + str(response_code))
        return
    var json = JSON.new()
    json.parse(body.get_string_from_utf8())
    var response = json.data
    if response.get("success", false):
        chat_response_received.emit(response["npc_name"], response["message"])
    else:
        chat_error.emit("对话失败")

func get_npc_status() -> void:
    """获取NPC背景对话状态（异步）"""
    if http_status.get_http_client_status() != HTTPClient.STATUS_DISCONNECTED:
        return  # 防止重复请求
    http_status.request(Config.API_NPC_STATUS)

func _on_status_request_completed(_result, response_code, _headers, body):
    if response_code != 200:
        return
    var json = JSON.new()
    json.parse(body.get_string_from_utf8())
    var response = json.data
    if response.has("dialogues"):
        npc_status_received.emit(response["dialogues"])
'''
print("api_client.gd (GDScript):\n" + api_client_gd_code)

### 15.6.2 对话 UI 实现

对话 UI 结构（图 15.13）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-13.png" alt="" width="85%"/>
  <p>图 15.13 对话 UI 结构</p>
</div>

In [ ]:
# dialogue_ui.gd — 对话 UI 脚本
dialogue_ui_gd_code = '''
extends CanvasLayer  # 始终显示在最上层

@onready var npc_name_label = $Panel/NPCName
@onready var npc_title_label = $Panel/NPCTitle
@onready var dialogue_text = $Panel/DialogueText   # RichTextLabel
@onready var input_field = $Panel/PlayerInput      # LineEdit
@onready var send_button = $Panel/SendButton
@onready var close_button = $Panel/CloseButton

var current_npc_name: String = ""

func _ready():
    visible = false
    send_button.pressed.connect(_on_send_button_pressed)
    close_button.pressed.connect(_on_close_button_pressed)
    input_field.text_submitted.connect(_on_text_submitted)  # 回车发送

func start_dialogue(npc_name: String):
    current_npc_name = npc_name
    npc_name_label.text = npc_name
    npc_title_label.text = get_npc_title(npc_name)
    dialogue_text.clear()
    dialogue_text.append_text("[color=gray]与 " + npc_name + " 的对话开始...[/color]\n")
    visible = true
    # 禁用玩家移动
    get_tree().get_first_node_in_group("player").set_interacting(true)
    input_field.grab_focus()

func send_message():
    var message = input_field.text.strip_edges()
    if message.is_empty() or current_npc_name.is_empty():
        return
    dialogue_text.append_text("\n[color=cyan]玩家:[/color] " + message + "\n")
    input_field.text = ""
    input_field.editable = false     # 等待响应时禁用输入
    send_button.disabled = true
    APIClient.send_chat(current_npc_name, message)  # 调用单例

func on_chat_response_received(npc_name: String, response: String):
    if npc_name == current_npc_name:
        dialogue_text.append_text("[color=yellow]" + npc_name + ":[/color] " + response + "\n")
        input_field.editable = true
        send_button.disabled = false
        input_field.grab_focus()

func get_npc_title(npc_name: String) -> String:
    var titles = {"张三": "Python工程师", "李四": "产品经理", "王五": "UI设计师"}
    return titles.get(npc_name, "")
'''
print("dialogue_ui.gd (GDScript):\n" + dialogue_ui_gd_code)

### 15.6.3 主场景整合

主场景脚本协调所有组件，定时从后端拉取 NPC 状态更新对话气泡。完整前后端通信流程（图 15.14）：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-14.png" alt="" width="85%"/>
  <p>图 15.14 前后端通信完整流程</p>
</div>

In [ ]:
# main.gd — 主场景脚本
main_gd_code = '''
extends Node2D

@onready var npc_zhang: Node2D = $NPCs/NPC_Zhang
@onready var npc_li: Node2D    = $NPCs/NPC_Li
@onready var npc_wang: Node2D  = $NPCs/NPC_Wang

var api_client: Node = null
var status_update_timer: float = 0.0

func _ready():
    api_client = get_node_or_null("/root/APIClient")
    if api_client:
        api_client.npc_status_received.connect(_on_npc_status_received)
        api_client.get_npc_status()  # 立即获取一次

func _process(delta: float):
    # 每30秒定时更新NPC背景对话
    status_update_timer += delta
    if status_update_timer >= Config.NPC_STATUS_UPDATE_INTERVAL:
        status_update_timer = 0.0
        if api_client:
            api_client.get_npc_status()

func _on_npc_status_received(dialogues: Dictionary):
    """收到NPC状态更新，刷新头顶对话气泡"""
    for npc_name in dialogues:
        var npc_node = get_npc_node(npc_name)
        if npc_node:
            npc_node.update_dialogue(dialogues[npc_name])

func get_npc_node(npc_name: String) -> Node2D:
    match npc_name:
        "张三": return npc_zhang
        "李四": return npc_li
        "王五": return npc_wang
        _: return null
'''
print("main.gd (GDScript):\n" + main_gd_code)

## 15.7 总结与展望

### 15.7.1 本章回顾

在本章中，我们完成了一个完整的 AI 小镇项目——赛博小镇。技术栈如图 15.15 所示：

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/15-figures/15-15.png" alt="" width="85%"/>
  <p>图 15.15 赛博小镇技术栈</p>
</div>

**核心收获：**

**（1）技术架构设计**：Godot（渲染/交互）+ FastAPI（API/状态）+ HelloAgents（智能/记忆）分层设计，各部分独立开发测试。

**（2）NPC 智能体系统**：用 `SimpleAgent` 为每个 NPC 创建独立实例，精心设计的系统提示词赋予鲜明性格。

**（3）记忆与好感度系统**：`WorkingMemory`（短期）+ `EpisodicMemory`（长期语义检索）双层记忆；五级好感度动态调整 NPC 行为。

**（4）批量生成 + 即时响应混合模式**：批量生成降低 API 成本，即时响应保证交互质量。

**（5）Godot 游戏场景**：场景模块化设计，玩家 WASD 控制，NPC 随机巡逻，信号机制松耦合通信。

**（6）前后端通信**：Godot HTTPRequest 异步请求 + 信号机制，保证游戏流畅性。

### 15.7.2 扩展方向

赛博小镇还有很多可以扩展的方向：

1. **多人在线支持**：WebSocket 实时通信，NPC 对每个玩家保持独立好感度
2. **任务系统**：好感度达到阈值后 NPC 提供特殊任务
3. **NPC 之间的互动**：张三和李四讨论需求，王五和张三讨论设计实现
4. **情感系统**：开心/难过/生气等情绪状态影响 NPC 回复风格
5. **动态事件系统**：团队会议、生日派对、紧急任务等
6. **更大的世界**：咖啡厅、图书馆、公园等多场景探索
7. **个性化学习**：NPC 记住玩家兴趣偏好，主动推送相关内容

### 15.7.3 思考与展望

赛博小镇展示了 AI 技术在游戏中的巨大潜力。AI NPC 面临的主要挑战是：API 成本（批量生成可缓解）、推理延迟（本地小模型是未来方向）、内容可控性（精心设计的提示词 + 内容过滤）。

随着 LLM 技术的发展，推理速度会越来越快，成本越来越低，本地化小型 LLM 也在快速发展。AI 技术与游戏的结合，将为玩家带来前所未有的体验。

**在第五部分的毕业设计章节，我们将学习如何用单智能体和多智能体构造通用智能体，这将是你的创作时间，敬请期待！**